# Model Evaluation Dashboard
Pick a **coin** and a **model** from the dropdowns, then click **Generate graphs**.
Runs in **Google Colab** OR **locally** (see the setup cell).
The model's trainer must have been run first (it saves a results bundle in `eval_bundles/`).

In [1]:
%pip install ta mplfinance matplotlib pandas scikit-learn ipywidgets numpy

import sys, os, importlib, numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error

LOCAL_DIR = None     # e.g. r'C:\\Users\\ofirz\\CryptoProject'

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/CryptoProject'
except ImportError:
    BASE_DIR = LOCAL_DIR if LOCAL_DIR else os.path.abspath(os.getcwd())

assert os.path.isdir(BASE_DIR), f'BASE_DIR not found: {BASE_DIR}  -> set LOCAL_DIR to your CryptoProject path'
sys.path.append(BASE_DIR)
import crypto_eval
importlib.reload(crypto_eval)
print('Ready. BASE_DIR =', BASE_DIR)
bdir = os.path.join(BASE_DIR, 'eval_bundles')
print('Available bundles:', sorted(os.listdir(bdir)) if os.path.isdir(bdir) else 'NONE - run a trainer first')

# ── Patch generate() to always support graphs= and n_plot= ──────────────────
import pandas as pd
try:
    import ta
    import mplfinance as mpf
    _mpf_ok = True
except ImportError:
    _mpf_ok = False

_ALL_GRAPHS = ['loss', 'prediction', 'direction', 'candles']

def generate(symbol, model_type='lstm', base_dir=None, graphs=None, n_plot=576,
             target_name='Bollinger %B'):
    base_dir = base_dir or BASE_DIR
    graphs = set(graphs) if graphs else set(_ALL_GRAPHS)

    path = os.path.join(base_dir, 'eval_bundles', f'{model_type}_{symbol}.npz')
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"No eval bundle for {symbol}/{model_type} at:\n  {path}\n"
            "Run that model's trainer first.")

    d = np.load(path)
    y_test      = np.asarray(d['y_test']).flatten()
    predictions = np.asarray(d['predictions']).flatten()
    train_losses = np.asarray(d['train_losses']).flatten() if d['train_losses'].size else np.array([])
    val_losses   = np.asarray(d['val_losses']).flatten()   if d['val_losses'].size  else np.array([])
    model_name  = 'TFT' if model_type.lower() in ('transformer','tft') else 'LSTM'

    # metrics
    rmse      = np.sqrt(mean_squared_error(y_test, predictions))
    corr      = np.corrcoef(predictions, y_test)[0, 1]
    base_rmse = np.sqrt(mean_squared_error(y_test[1:], y_test[:-1]))
    base_corr = np.corrcoef(y_test[:-1], y_test[1:])[0, 1]
    improvement = (base_rmse - rmse) / base_rmse * 100
    corr_gain   = corr - base_corr
    actual_dir  = np.sign(np.diff(y_test))
    pred_dir    = np.sign(np.diff(predictions))
    mask        = actual_dir != 0
    dir_acc     = (pred_dir[mask] == actual_dir[mask]).mean() * 100 if mask.sum() else float('nan')

    print(f"\n--- Results ({symbol} {model_name}, target: {target_name}) ---")
    print(f"{'':22}{'MODEL':>12}{'PERSISTENCE':>14}")
    print(f"{'RMSE':22}{rmse:>12.6f}{base_rmse:>14.6f}")
    print(f"{'Correlation':22}{corr:>12.3f}{base_corr:>14.3f}")
    print(f"\nRMSE vs persistence baseline: {improvement:+.1f}%")
    print(f"Correlation GAIN over pure copying: {corr_gain:+.3f}")
    print(f"Directional accuracy: {dir_acc:.1f}%  (50% = coin flip)")

    # 1) Training loss
    if 'loss' in graphs and train_losses.size:
        plt.figure(figsize=(10, 5))
        plt.plot(train_losses, label='Train Loss')
        plt.plot(val_losses,   label='Validation Loss')
        plt.title(f'{symbol} {model_name} ({target_name}) Training Process')
        plt.xlabel('Epochs'); plt.ylabel('Loss')
        plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

    # 2) Actual vs Predicted
    if 'prediction' in graphs:
        k = min(n_plot, len(y_test))
        plt.figure(figsize=(14, 7))
        plt.plot(y_test[:k],      label=f'Actual {target_name}',    color='blue', alpha=0.7)
        plt.plot(predictions[:k], label=f'Predicted {target_name}', color='red',  linewidth=1.2)
        plt.title(f'{symbol} {model_name} Prediction Performance ({target_name})\n'
                  f'RMSE={rmse:.5f} | Corr={corr:.2f} | vs baseline {improvement:+.1f}%')
        plt.xlabel('Time Steps (5m intervals)'); plt.ylabel(target_name)
        plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

    # 3) Directional accuracy bar
    if 'direction' in graphs:
        plt.figure(figsize=(5, 5))
        bars = plt.bar(['Model', 'Coin flip'], [dir_acc, 50.0], color=['#2a9d8f', '#bbbbbb'])
        plt.axhline(50, color='gray', linestyle='--', linewidth=0.8)
        plt.ylim(0, 100); plt.ylabel('Directional Accuracy (%)')
        plt.title(f'{symbol} {model_name} Directional Accuracy')
        for b in bars:
            plt.text(b.get_x() + b.get_width() / 2, b.get_height() + 1,
                     f'{b.get_height():.1f}%', ha='center', va='bottom')
        plt.tight_layout(); plt.show()

    # 4) Candle chart
    if 'candles' in graphs and _mpf_ok:
        csv_path = os.path.join(base_dir, 'data', f'{symbol}_5m_data.csv')
        try:
            df_raw = pd.read_csv(csv_path)
            bb = ta.volatility.BollingerBands(df_raw['close'], window=20, window_dev=2)
            upper = bb.bollinger_hband().values
            lower = bb.bollinger_lband().values
            test_offset = len(df_raw) - len(predictions)
            df_c = df_raw.iloc[test_offset:].copy()
            df_c['open_time'] = pd.to_datetime(df_c['open_time'])
            df_c = df_c.set_index('open_time')
            pred_price = lower[test_offset:] + (predictions * (upper[test_offset:] - lower[test_offset:]))
            df_c['Predicted_Close'] = pd.Series(pred_price, index=df_c.index).rolling(3).mean().bfill()
            df_plot = df_c.head(n_plot)
            ap = mpf.make_addplot(df_plot['Predicted_Close'], type='line', color='red',
                                  linestyle='--', width=1.5, panel=0)
            mpf.plot(df_plot, type='candle', style='yahoo', addplot=ap, volume=True,
                     title=f'{symbol} {model_name} Prediction vs Actual\nRMSE ({target_name}): {rmse:.5f}',
                     ylabel='Price (USDT)', figsize=(14, 8), tight_layout=True)
        except Exception as e:
            print(f"(Candle plot skipped: {e})")

    return {'rmse': rmse, 'corr': corr, 'improvement': improvement,
            'corr_gain': corr_gain, 'dir_acc': dir_acc}


Mounted at /content/drive
Ready. BASE_DIR = /content/drive/MyDrive/CryptoProject
Available bundles: ['lstm_BTCUSDT.npz', 'lstm_ETHUSDT.npz', 'lstm_XRPUSDT.npz', 'transformer_BTCUSDT.npz', 'transformer_ETHUSDT.npz', 'transformer_XRPUSDT.npz']


In [2]:
# ============================================================
#   CHOOSE coin + model + graphs, then click the button
# ============================================================
_GRAPH_OPTIONS = {
    'Training Loss':        'loss',
    'Actual vs Predicted':  'prediction',
    'Directional Accuracy': 'direction',
    'Candle Chart':         'candles',
}

coin_dd  = widgets.Dropdown(
    options=['BTCUSDT', 'ETHUSDT', 'XRPUSDT'], value='BTCUSDT', description='Coin:')
model_dd = widgets.Dropdown(
    options=['lstm', 'transformer'], value='lstm', description='Model:')

n_plot_input = widgets.BoundedIntText(
    value=576, min=50, max=10000, step=50,
    description='# Points:',
    layout=widgets.Layout(width='180px'),
    style={'description_width': '70px'},
)

points_info = widgets.HTML(value='')   # dynamic hint under # Points

def _update_points_info(*_):
    path = os.path.join(BASE_DIR, 'eval_bundles', f'{model_dd.value}_{coin_dd.value}.npz')
    try:
        d = np.load(path)
        total = len(np.asarray(d['y_test']).flatten())
        minutes = total * 5
        hours   = minutes / 60
        days    = hours / 24
        if days >= 1:
            time_str = f'~{days:.0f} days'
        else:
            time_str = f'~{hours:.0f} hours'
        n_plot_input.max = total
        points_info.value = (
            f'<span style="font-size:12px;color:#555;">'
            f'<b>Test set:</b> {total:,} points ({time_str} of 5-min candles).<br>'
            f'Each point = 1 candle (5 min). '
            f'<b># Points</b> zooms the chart — max is the full test set ({total:,}).'
            f'</span>'
        )
    except FileNotFoundError:
        n_plot_input.max = 10000
        points_info.value = (
            '<span style="font-size:12px;color:#999;">'
            'No bundle found for this coin/model — run the trainer first.<br>'
            '<b># Points</b> zooms the chart (each point = one 5-min candle).'
            '</span>'
        )

coin_dd.observe(_update_points_info, names='value')
model_dd.observe(_update_points_info, names='value')
_update_points_info()   # populate on load

# One checkbox per graph type — all ticked by default
graph_checks = {
    label: widgets.Checkbox(value=True, description=label, indent=False,
                            layout=widgets.Layout(width='220px'))
    for label in _GRAPH_OPTIONS
}

run_btn = widgets.Button(description='Generate graphs', button_style='success', icon='chart-line')
out     = widgets.Output()

def _on_click(_):
    with out:
        clear_output(wait=True)
        selected_ids = [_GRAPH_OPTIONS[lbl] for lbl, cb in graph_checks.items() if cb.value]
        if not selected_ids:
            print("Tick at least one graph type.")
            return
        try:
            generate(coin_dd.value, model_dd.value, base_dir=BASE_DIR,
                     graphs=selected_ids, n_plot=n_plot_input.value)
        except FileNotFoundError as e:
            print(e)

run_btn.on_click(_on_click)

top_row    = widgets.HBox([coin_dd, model_dd,
                           widgets.VBox([n_plot_input, points_info]),
                           run_btn],
                          layout=widgets.Layout(align_items='flex-start', gap='12px'))
checks_box = widgets.VBox(list(graph_checks.values()),
                          layout=widgets.Layout(margin='6px 0 0 0'))
display(widgets.VBox([top_row, checks_box]), out)


Output()

In [3]:
if False:
    generate('BTCUSDT','lstm' , base_dir=BASE_DIR)
    generate('BTCUSDT','transformer' , base_dir=BASE_DIR)
    generate('ETHUSDT','lstm' , base_dir=BASE_DIR)
    generate('ETHUSDT','transformer' , base_dir=BASE_DIR)
    generate('XRPUSDT','lstm' , base_dir=BASE_DIR)
    generate('XRPUSDT','transformer' , base_dir=BASE_DIR)
